Imports


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

How we should count delayed? If NA is present should we assume that there was no delay? Below I dropped the NA values and did some basic calculations.
-ANSWER-
For NA values we can assume there was no delay

In [ ]:
path = '../../data/kaggle/flights_sample_100k.csv'
flights = pd.read_csv(path)
flights

In [ ]:
flights.shape

In [ ]:
flights.info()

In [ ]:
flights.describe()

In [ ]:
flights.isnull().sum()

For delay related columns we can fill them with '0' for NA

In [ ]:
delay_cols = ['DELAY_DUE_CARRIER','DELAY_DUE_WEATHER','DELAY_DUE_NAS','DELAY_DUE_SECURITY',	'DELAY_DUE_LATE_AIRCRAFT']
flights[delay_cols] = flights[delay_cols].fillna(0.0)
flights

In [ ]:
flights_delayed_info = flights[['FL_DATE','AIRLINE','AIRLINE_CODE','DOT_CODE','DELAY_DUE_CARRIER','DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']]
flights_delayed_info

In [ ]:
flights_delayed_info = flights_delayed_info.dropna().reset_index()
flights_delayed_info

In [ ]:
flights_delayed_info['WAS_DELAYED'] = (flights_delayed_info[delay_cols] > 0).any(axis=1).astype(int)

In [ ]:
flights_delayed_info

In [ ]:
print(f"Percentage: {flights_delayed_info['WAS_DELAYED'].sum()/len(flights_delayed_info)*100:.2f}% were delayed")

In [ ]:
delay_per_cols_counts = (flights[delay_cols]>0).sum().reset_index()

In [ ]:
delay_per_cols_counts.columns = ["Delay_Type", "Flights"]

# Plot
plt.figure(figsize=(8,5))

sns.barplot(
    data=delay_per_cols_counts,
    x="Delay_Type",
    y="Flights",
    palette="viridis",
    legend=False
)

plt.title("Number of Flights by Delay Type")
plt.xlabel("Delay Type")
plt.ylabel("Number of Flights")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Combine weather with flights dataset

In [ ]:
from weather_utils import get_weather_for_flights, save_and_optimize_weather_ds

flights_df_with_weather = get_weather_for_flights(flights)

In [ ]:
save_and_optimize_weather_ds(flights_df_with_weather)

In [ ]:
def finalize_flight_weather_dataset(df):
    # Zamieniamy np. 1530.0 na '15:30'
    df['tmp_time'] = df['CRS_DEP_TIME'].astype(int).astype(str).str.zfill(4)
    df['tmp_time'] = df['tmp_time'].str[:2] + ':' + df['tmp_time'].str[2:]
    
    # Używamy .dt.round('h'), pogoda jest pobrana co godzine
    df['scheduled_datetime'] = pd.to_datetime(df['FL_DATE'].astype(str) + ' ' + df['tmp_time'], errors='coerce')
    df['weather_hour_key'] = df['scheduled_datetime'].dt.round('h').dt.tz_localize(None)
    df.drop(columns=['tmp_time'], inplace=True)
    
    return df

In [ ]:
flights_df_with_weather = finalize_flight_weather_dataset(flights_df_with_weather)
flights_df_with_weather

In [ ]:
flights_df_with_weather.columns

In [ ]:
from weather_utils import export_master_weather
export_master_weather(flights_df_with_weather)

Test dla innego datasetu by wziął dane z parqueta

In [ ]:
from weather_manager import sync_weather_repository, clean_delay_columns
path = '../../data/kaggle/flights_sample_10k.csv'
flights_2k = pd.read_csv(path)
flights_2k = clean_delay_columns(flights_2k)

In [ ]:
sync_weather_repository(flights_2k)